# 딥러닝 미니프로젝트

In [1]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================
import os
# [중요] Intel OpenMP 라이브러리 중복 로드 충돌을 방지하기 위한 환경 설정
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import torch                          # 텐서 연산, 자동 미분 등 기본 기능
import torch.nn as nn                 # 신경망 레이어 모음 (Linear, ReLU 등)
import torch.optim as optim           # 옵티마이저 모음 (SGD, Adam 등)
import torch.nn.functional as F       # 활성화 함수 등 함수형 인터페이스

# 데이터 처리
from torch.utils.data import DataLoader, TensorDataset
import torchvision                    # 이미지 데이터셋 제공 (MNIST 등)
import torchvision.transforms as transforms  # 이미지 전처리 변환

# 수학 / 시각화
import numpy as np                    # 수치 계산
import matplotlib.pyplot as plt       # 그래프 그리기
import pandas as pd
import seaborn as sns
import matplotlib
# 한글 폰트 설정 (운영체제별로 다름)
import platform
if platform.system() == 'Darwin':     # macOS
    matplotlib.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':  # Windows
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
else:                                 # Linux
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import random
import os

def set_seed(seed=42):
    # 1. 기본 Python 및 OS 시드 고정
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # 2. PyTorch 시드 고정 (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용 시
    
    # 3. CUDA 연산의 결정론적 설정 (속도는 조금 느려질 수 있음)
    # CuDNN이 최적의 알고리즘을 찾는 과정을 멈추고 항상 같은 알고리즘을 쓰게 함
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✅ 모든 시드가 {seed}로 고정되었습니다.")

set_seed(42)


# ── GPU vs CPU 자동 선택 ─────────────────────────────────────────────────────
# GPU가 있으면 GPU(CUDA)를, 없으면 CPU를 사용
# GPU: 병렬 연산이 가능하여 딥러닝 학습 속도가 수십~수백 배 빠름
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 장치: {device}')
print(f'PyTorch 버전: {torch.__version__}')
print(f'GPU 사용 가능: {torch.cuda.is_available()}')

✅ 모든 시드가 42로 고정되었습니다.
사용 장치: cuda
PyTorch 버전: 2.12.0.dev20260323+cu128
GPU 사용 가능: True


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

def get_mean_std(loader):
    # 채널별 합계, 제곱합, 전체 픽셀 수 초기화
    channels_sum, channels_squared_sum, num_batches = 0, 0, 0
    
    for data, _ in loader:
        # data: [batch_size, 3, height, width]
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_squared_sum += torch.mean(data**2, dim=[0, 2, 3])
        num_batches += 1
    
    mean = channels_sum / num_batches
    std = (channels_squared_sum / num_batches - mean**2)**0.5
    
    return mean, std

# 계산을 위한 임시 로더 (사이즈만 통일하고 텐서로 변환)
temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Train 폴더 기준 계산 (전체 데이터셋의 경향성을 대표함)
temp_dataset = datasets.ImageFolder(root='./Face Mask Dataset/Train', transform=temp_transform)
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=False, num_workers=4)

print("Calculating Mean and Std... (Please wait)")
mean, std = get_mean_std(temp_loader)
print(f"Calculated Mean: {mean}")
print(f"Calculated Std: {std}")

Calculating Mean and Std... (Please wait)
Calculated Mean: tensor([0.5688, 0.4649, 0.4161])
Calculated Std: tensor([0.2821, 0.2587, 0.2603])


In [4]:
# 실제 학습에 사용할 Final Transform
final_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # 팀 프로젝트 팁: 훈련 데이터에는 랜덤 뒤집기 등을 추가하면 성능이 더 좋아집니다.
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist()) # 직접 구한 값 적용!
])

# 데이터셋 로드
train_set = datasets.ImageFolder(root='./Face Mask Dataset/Train', transform=final_transform)
val_set = datasets.ImageFolder(root='./Face Mask Dataset/Validation', transform=final_transform)
test_set = datasets.ImageFolder(root='./Face Mask Dataset/Test', transform=final_transform)

# 4000 시리즈 GPU(RTX 5060) 최적화 설정
# pin_memory=True를 설정하면 CPU에서 GPU로 데이터 전송 속도가 빨라집니다.
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_set)}, Validation samples: {len(val_set)},Test Samples: {len(test_set)}")

Train samples: 10000, Validation samples: 800,Test Samples: 992


In [5]:
# 클래스와 인덱스 매핑 확인
print(f"클래스 이름: {train_set.classes}")
print(f"클래스별 인덱스: {train_set.class_to_idx}")

# 데이터 로더에서 한 배지만 꺼내서 확인
images, labels = next(iter(train_loader))
print(f"배치 내 이미지 모양: {images.shape}") # [32, 3, 224, 224]
print(f"배치 내 라벨들: {labels}") # 0과 1이 섞여서 나옴

클래스 이름: ['WithMask', 'WithoutMask']
클래스별 인덱스: {'WithMask': 0, 'WithoutMask': 1}
배치 내 이미지 모양: torch.Size([32, 3, 224, 224])
배치 내 라벨들: tensor([1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1,
        0, 0, 1, 1, 1, 1, 0, 1])
